<a href="https://colab.research.google.com/github/cerrutisofi/TP-FINAL-DATOS/blob/main/Presupuesto_Participativo_Borrador.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Importación y ejecución de las librerías de Python para el análisis,
# visualización y modelado de datos.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, log_loss
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV

In [ ]:
df = pd.read_csv('https://github.com/cerrutisofi/TP-FINAL-DATOS/raw/refs/heads/main/DATOS/Datos%20procesados/dataset_final_vilo.csv', sep=';', header=0)

df


,Año,Localidades,Proyecto,Categoria,Votos,Presupuesto,Ganador_bin
0,2017,Villa Martelli,BOMBEROS VOLUNTARIOS DE VICENTE LÓPEZ - 25 equ...,Seguridad,959,978500,1
1,2017,Villa Martelli,CAMPAÑA DE CONCIENTIZACIÓN SOBRE EL BULLYING -...,Cultura,506,193892,1
2,2017,Villa Martelli,FUNDACION CAMINO - Remodelación interior y fac...,Reparaciones/Obra,487,1500000,1
3,2017,Villa Martelli,ESCUELA SECUNDARIA Nº 2 (provincial) - Reparac...,Educación,482,1200000,1
4,2017,Villa Martelli,BIBLIOTECA FROILÁN GONZÁLEZ - Compra de mobili...,Cultura,446,132000,1
...,...,...,...,...,...,...,...
1523,2013,Olivos,Mejoramiento de iluminación,Seguridad,540,486200,1
1524,2013,Vicente López,Reducidores de velocidad,Movilidad,540,14164,1
1525,2013,Vicente López,Cámaras de vigilancia urbana conectadas al Cen...,Seguridad,540,50000,1
1526,2013,Vicente López,PISTA BMX,Deporte,540,147000,1


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1528 entries, 0 to 1527
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Año          1528 non-null   int64 
 1   Localidades  1528 non-null   object
 2   Proyecto     1528 non-null   object
 3   Categoria    1528 non-null   object
 4   Votos        1528 non-null   int64 
 5   Presupuesto  1528 non-null   int64 
 6   Ganador_bin  1528 non-null   int64 
dtypes: int64(4), object(3)
memory usage: 83.7+ KB


In [ ]:
df.value_counts('Categoria')

,count
Categoria,
Educación,322
Social,236
Movilidad,163
Cultura,137
Deporte,134
Salud,132
Seguridad,112
Medio Ambiente,101
Espacio Público,69


In [ ]:
df.value_counts('Localidades')

,count
Localidades,
Olivos,261
Munro,237
Florida Oeste,196
Villa Martelli,191
Florida Este,191
Carapachay,126
Vicente López,126
Villa Adelina,106
La Lucila,94


In [ ]:
df_censo_2022 = pd.read_excel('https://github.com/cerrutisofi/TP-FINAL-DATOS/raw/refs/heads/main/DATOS/Datos%20procesados/Censo%202022%20-%20Vicente%20L%C3%B3pez.xlsx')

df_censo_2022


,Municipios,Localidades,Geolocalización,Límites,Viviendas particulares,Población en viviendas particulares,Mujer/Femenino,Varón/Masculino,0-4,5-9,...,No tiene internet en la vivienda,Obra social o\nprepaga\n(incluye PAMI),Programas o\nplanes estatales\nde salud,"No tiene obra\nsocial, prepaga ni\nplan estatal",Población\ncon asistencia escolar,Población\nsin asistencia escolar\npero que si asistió,Población\nque nunca\ntuvo asistencia escolar,Población económicamente activa ocupada,Población económicamente activa desocupada,Población no económicamente activa
0,Vicente López,Carapachay,"-58.5452628,-34.5277154,0.0 -58.5388899,-34.53...",Polygon Outer Boundary,6995,16914,8910,8004,681,939,...,597,14271,227,2416,5131,11463,320,8507,722,5184
1,Vicente López,Florida Este,"-58.4935926,-34.5466773,0.0 -58.4758687,-34.53...",Polygon Outer Boundary,22270,49536,26765,22771,2083,2563,...,1456,44666,733,4137,15196,32648,1692,25955,2311,14243
2,Vicente López,Florida Oeste,"-58.5288477,-34.539086,0.0 -58.5286761,-34.540...",Polygon Outer Boundary,11524,28227,14713,13514,1268,1889,...,1284,20721,650,6856,9021,18445,761,14289,1237,7936
3,Vicente López,La Lucila,"-58.4974766,-34.4975558,0.0 -58.492949,-34.506...",Polygon Outer Boundary,6321,13554,7236,6318,609,710,...,341,12426,176,952,4000,8923,631,6897,646,4077
4,Vicente López,Munro,"-58.5248244,-34.5111098,0.0 -58.5308325,-34.51...",Polygon Outer Boundary,14580,34797,18277,16520,1502,2037,...,1548,26894,640,7263,10561,23259,977,17970,1656,9979
5,Vicente López,Olivos,"-58.492949,-34.5063622,0.0 -58.4974122,-34.497...",Polygon Outer Boundary,36530,77296,41612,35684,3189,4075,...,2351,68629,1068,7599,22946,52574,1776,40877,2988,22731
6,Vicente López,Vicente López,"-58.4865976,-34.5183323,0.0 -58.4758687,-34.53...",Polygon Outer Boundary,12830,24790,13371,11419,1080,1188,...,665,22935,280,1575,7292,16669,829,13177,844,7418
7,Vicente López,Villa Adelina,"-58.5398769,-34.516246,0.0 -58.5552191,-34.529...",Polygon Outer Boundary,3657,8707,4609,4098,334,498,...,327,7162,139,1406,2665,5787,255,4455,408,2587
8,Vicente López,Villa Martelli,"-58.4999871,-34.5388297,0.0 -58.5248351,-34.55...",Polygon Outer Boundary,11030,26720,14017,12703,1210,1609,...,1215,20368,580,5772,8290,17336,1094,13358,1249,7923


In [ ]:
df_censo_2022.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 46 columns):
 #   Column                                                                 Non-Null Count  Dtype 
---  ------                                                                 --------------  ----- 
 0   Municipios                                                             9 non-null      object
 1   Localidades                                                            9 non-null      object
 2   Geolocalización                                                        9 non-null      object
 3   Límites                                                                9 non-null      object
 4   Viviendas particulares                                                 9 non-null      int64 
 5   Población en viviendas particulares                                    9 non-null      int64 
 6   Mujer/Femenino                                                         9 non-null      int64 
 7   Var

In [ ]:
df_censo_2022.value_counts('Perforación con bomba manual')

,count
Perforación con bomba manual,
4,2
7,2
2,1
11,1
19,1
39,1
-,1


In [ ]:
df_censo_2022.value_counts('Pozo sin bomba')

,count
Pozo sin bomba,
-,3
3,2
9,1
11,1
13,1
25,1


In [ ]:
columnas = ['Perforación con bomba manual', 'Pozo sin bomba']

for col in columnas:
    # Transformación de los valores que no sean números en NaN (Not a Number)
    df_censo_2022[col] = pd.to_numeric(df_censo_2022[col], errors='coerce')

    # 2. Reemplazo de los NaN con 0
    df_censo_2022[col] = df_censo_2022[col].fillna(0)

    # 3. Transformación de los valores de object a int64
    df_censo_2022[col] = df_censo_2022[col].astype('int64')

In [ ]:
df_geolocalización = pd.read_excel('https://github.com/cerrutisofi/TP-FINAL-DATOS/raw/refs/heads/main/DATOS/Datos%20procesados/Geolocalizaci%C3%B3n%20Vicente%20L%C3%B3pez%20-%20Unificado.xlsx')

df_geolocalización

,Nombre,Categoria,Latitud,Longitud,Dirección
0,Delegación CARAPACHAY,Residuos Especiales Domiciarios,-58.5357028,-34.52625832,Independencia y Ascasubi
1,Delegación FLORIDA,Residuos Especiales Domiciarios,-58.48407961,-34.52454026,Juan B. Justo 1379
2,Delegación FLORIDA OESTE,Residuos Especiales Domiciarios,-58.51494301,-34.53938955,Av. San Martín 4106
3,Delegación MUNRO,Residuos Especiales Domiciarios,-58.51947107,-34.52765363,Av. Velez Sarsfield 4839
4,Destacamento Plaza La Paz,Residuos Especiales Domiciarios,NaN,NaN,"C. Saavedra 1940, FLORIDA OESTE"
...,...,...,...,...,...
62,CAPS Blanca Acosta,Centros de Salud,-34.5464338,-58.5241343,"Gervasio Posadas 1276, Florida Oeste."
63,CAPS Senador Bermúdez,Centros de Salud,-34.5392593,-58.52549,"Hipólito Yrigoyen 4601, Florida Oeste."
64,CAPS Llobera,Centros de Salud,"-34,5468977","-58,5051814","Estados Unidos 314, Villa Martelli."
65,CAPS Ravazzoli,Centros de Salud,-34.5509299,58.5209154,"Moldes 4900, Villa Martelli."


In [ ]:
df_geolocalización.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67 entries, 0 to 66
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Nombre     67 non-null     object
 1   Categoria  67 non-null     object
 2   Latitud    66 non-null     object
 3   Longitud   66 non-null     object
 4   Dirección  67 non-null     object
dtypes: object(5)
memory usage: 2.7+ KB


In [ ]:
# Chequear los tipos de datos de cada df y corregir los que correspondan

# Realizar una agrupación de los barrios con sus correspondientes localidades

# Realizar un join entre los dos primeros dataframes

In [ ]:
# Unión de los dos primeros dataframes

df_definitivo = pd.merge(df, df_censo_2022, on='Localidades', how='inner')

df_definitivo

,Año,Localidades,Proyecto,Categoria,Votos,Presupuesto,Ganador_bin,Municipios,Geolocalización,Límites,...,No tiene internet en la vivienda,Obra social o\nprepaga\n(incluye PAMI),Programas o\nplanes estatales\nde salud,"No tiene obra\nsocial, prepaga ni\nplan estatal",Población\ncon asistencia escolar,Población\nsin asistencia escolar\npero que si asistió,Población\nque nunca\ntuvo asistencia escolar,Población económicamente activa ocupada,Población económicamente activa desocupada,Población no económicamente activa
0,2017,Villa Martelli,BOMBEROS VOLUNTARIOS DE VICENTE LÓPEZ - 25 equ...,Seguridad,959,978500,1,Vicente López,"-58.4999871,-34.5388297,0.0 -58.5248351,-34.55...",Polygon Outer Boundary,...,1215,20368,580,5772,8290,17336,1094,13358,1249,7923
1,2017,Villa Martelli,CAMPAÑA DE CONCIENTIZACIÓN SOBRE EL BULLYING -...,Cultura,506,193892,1,Vicente López,"-58.4999871,-34.5388297,0.0 -58.5248351,-34.55...",Polygon Outer Boundary,...,1215,20368,580,5772,8290,17336,1094,13358,1249,7923
2,2017,Villa Martelli,FUNDACION CAMINO - Remodelación interior y fac...,Reparaciones/Obra,487,1500000,1,Vicente López,"-58.4999871,-34.5388297,0.0 -58.5248351,-34.55...",Polygon Outer Boundary,...,1215,20368,580,5772,8290,17336,1094,13358,1249,7923
3,2017,Villa Martelli,ESCUELA SECUNDARIA Nº 2 (provincial) - Reparac...,Educación,482,1200000,1,Vicente López,"-58.4999871,-34.5388297,0.0 -58.5248351,-34.55...",Polygon Outer Boundary,...,1215,20368,580,5772,8290,17336,1094,13358,1249,7923
4,2017,Villa Martelli,BIBLIOTECA FROILÁN GONZÁLEZ - Compra de mobili...,Cultura,446,132000,1,Vicente López,"-58.4999871,-34.5388297,0.0 -58.5248351,-34.55...",Polygon Outer Boundary,...,1215,20368,580,5772,8290,17336,1094,13358,1249,7923
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1523,2013,Olivos,Mejoramiento de iluminación,Seguridad,540,486200,1,Vicente López,"-58.492949,-34.5063622,0.0 -58.4974122,-34.497...",Polygon Outer Boundary,...,2351,68629,1068,7599,22946,52574,1776,40877,2988,22731
1524,2013,Vicente López,Reducidores de velocidad,Movilidad,540,14164,1,Vicente López,"-58.4865976,-34.5183323,0.0 -58.4758687,-34.53...",Polygon Outer Boundary,...,665,22935,280,1575,7292,16669,829,13177,844,7418
1525,2013,Vicente López,Cámaras de vigilancia urbana conectadas al Cen...,Seguridad,540,50000,1,Vicente López,"-58.4865976,-34.5183323,0.0 -58.4758687,-34.53...",Polygon Outer Boundary,...,665,22935,280,1575,7292,16669,829,13177,844,7418
1526,2013,Vicente López,PISTA BMX,Deporte,540,147000,1,Vicente López,"-58.4865976,-34.5183323,0.0 -58.4758687,-34.53...",Polygon Outer Boundary,...,665,22935,280,1575,7292,16669,829,13177,844,7418


In [ ]:
df_definitivo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1528 entries, 0 to 1527
Data columns (total 52 columns):
 #   Column                                                                 Non-Null Count  Dtype 
---  ------                                                                 --------------  ----- 
 0   Año                                                                    1528 non-null   int64 
 1   Localidades                                                            1528 non-null   object
 2   Proyecto                                                               1528 non-null   object
 3   Categoria                                                              1528 non-null   object
 4   Votos                                                                  1528 non-null   int64 
 5   Presupuesto                                                            1528 non-null   int64 
 6   Ganador_bin                                                            1528 non-null   int64 
 7

In [ ]:
# Creación de las nuevas columnas unificadas

df_definitivo['0-14'] = df_definitivo['0-4'] + df_definitivo['5-9'] + df_definitivo['10-14']

df_definitivo['15-64'] = df_definitivo['15-19'] + df_definitivo['20-24'] + df_definitivo['25-29'] + df_definitivo['30-34'] + df_definitivo['35-39'] + df_definitivo['40-44'] + df_definitivo['45-49'] + df_definitivo['50-54'] + df_definitivo['55-59'] + df_definitivo['60-64']

df_definitivo['65-100 y más'] = df_definitivo['65-69'] + df_definitivo['70-74'] + df_definitivo['75-79'] + df_definitivo['80-84'] + df_definitivo['85-89'] + df_definitivo['90-94'] + df_definitivo['95-99'] + df_definitivo['100 y más']

# Eliminación de las columnas originales que ya no se necesitan

df_definitivo = df_definitivo.drop(columns=['0-4', '5-9', '10-14', '15-19', '20-24', '25-29', '30-34', '35-39', '40-44', '45-49', '50-54', '55-59', '60-64', '65-69', '70-74', '75-79', '80-84', '85-89', '90-94', '95-99', '100 y más'])

df_definitivo

,Año,Localidades,Proyecto,Categoria,Votos,Presupuesto,Ganador_bin,Municipios,Geolocalización,Límites,...,"No tiene obra\nsocial, prepaga ni\nplan estatal",Población\ncon asistencia escolar,Población\nsin asistencia escolar\npero que si asistió,Población\nque nunca\ntuvo asistencia escolar,Población económicamente activa ocupada,Población económicamente activa desocupada,Población no económicamente activa,0-14,15-64,65-100 y más
0,2017,Villa Martelli,BOMBEROS VOLUNTARIOS DE VICENTE LÓPEZ - 25 equ...,Seguridad,959,978500,1,Vicente López,"-58.4999871,-34.5388297,0.0 -58.5248351,-34.55...",Polygon Outer Boundary,...,5772,8290,17336,1094,13358,1249,7923,4547,17473,4700
1,2017,Villa Martelli,CAMPAÑA DE CONCIENTIZACIÓN SOBRE EL BULLYING -...,Cultura,506,193892,1,Vicente López,"-58.4999871,-34.5388297,0.0 -58.5248351,-34.55...",Polygon Outer Boundary,...,5772,8290,17336,1094,13358,1249,7923,4547,17473,4700
2,2017,Villa Martelli,FUNDACION CAMINO - Remodelación interior y fac...,Reparaciones/Obra,487,1500000,1,Vicente López,"-58.4999871,-34.5388297,0.0 -58.5248351,-34.55...",Polygon Outer Boundary,...,5772,8290,17336,1094,13358,1249,7923,4547,17473,4700
3,2017,Villa Martelli,ESCUELA SECUNDARIA Nº 2 (provincial) - Reparac...,Educación,482,1200000,1,Vicente López,"-58.4999871,-34.5388297,0.0 -58.5248351,-34.55...",Polygon Outer Boundary,...,5772,8290,17336,1094,13358,1249,7923,4547,17473,4700
4,2017,Villa Martelli,BIBLIOTECA FROILÁN GONZÁLEZ - Compra de mobili...,Cultura,446,132000,1,Vicente López,"-58.4999871,-34.5388297,0.0 -58.5248351,-34.55...",Polygon Outer Boundary,...,5772,8290,17336,1094,13358,1249,7923,4547,17473,4700
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1523,2013,Olivos,Mejoramiento de iluminación,Seguridad,540,486200,1,Vicente López,"-58.492949,-34.5063622,0.0 -58.4974122,-34.497...",Polygon Outer Boundary,...,7599,22946,52574,1776,40877,2988,22731,11587,50064,15645
1524,2013,Vicente López,Reducidores de velocidad,Movilidad,540,14164,1,Vicente López,"-58.4865976,-34.5183323,0.0 -58.4758687,-34.53...",Polygon Outer Boundary,...,1575,7292,16669,829,13177,844,7418,3631,15829,5330
1525,2013,Vicente López,Cámaras de vigilancia urbana conectadas al Cen...,Seguridad,540,50000,1,Vicente López,"-58.4865976,-34.5183323,0.0 -58.4758687,-34.53...",Polygon Outer Boundary,...,1575,7292,16669,829,13177,844,7418,3631,15829,5330
1526,2013,Vicente López,PISTA BMX,Deporte,540,147000,1,Vicente López,"-58.4865976,-34.5183323,0.0 -58.4758687,-34.53...",Polygon Outer Boundary,...,1575,7292,16669,829,13177,844,7418,3631,15829,5330


In [ ]:
df_definitivo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1528 entries, 0 to 1527
Data columns (total 34 columns):
 #   Column                                                                 Non-Null Count  Dtype 
---  ------                                                                 --------------  ----- 
 0   Año                                                                    1528 non-null   int64 
 1   Localidades                                                            1528 non-null   object
 2   Proyecto                                                               1528 non-null   object
 3   Categoria                                                              1528 non-null   object
 4   Votos                                                                  1528 non-null   int64 
 5   Presupuesto                                                            1528 non-null   int64 
 6   Ganador_bin                                                            1528 non-null   int64 
 7

In [ ]:
# Eliminamos las columnas 'Municipios','Geolocalización','Límites' ya que
# no generan valor al df_definitivo al momento del modelado de datos.

df_definitivo.drop(columns=['Proyecto','Municipios','Geolocalización','Límites'], inplace=True)

df_definitivo

,Año,Localidades,Categoria,Votos,Presupuesto,Ganador_bin,Viviendas particulares,Población en viviendas particulares,Mujer/Femenino,Varón/Masculino,...,"No tiene obra\nsocial, prepaga ni\nplan estatal",Población\ncon asistencia escolar,Población\nsin asistencia escolar\npero que si asistió,Población\nque nunca\ntuvo asistencia escolar,Población económicamente activa ocupada,Población económicamente activa desocupada,Población no económicamente activa,0-14,15-64,65-100 y más
0,2017,Villa Martelli,Seguridad,959,978500,1,11030,26720,14017,12703,...,5772,8290,17336,1094,13358,1249,7923,4547,17473,4700
1,2017,Villa Martelli,Cultura,506,193892,1,11030,26720,14017,12703,...,5772,8290,17336,1094,13358,1249,7923,4547,17473,4700
2,2017,Villa Martelli,Reparaciones/Obra,487,1500000,1,11030,26720,14017,12703,...,5772,8290,17336,1094,13358,1249,7923,4547,17473,4700
3,2017,Villa Martelli,Educación,482,1200000,1,11030,26720,14017,12703,...,5772,8290,17336,1094,13358,1249,7923,4547,17473,4700
4,2017,Villa Martelli,Cultura,446,132000,1,11030,26720,14017,12703,...,5772,8290,17336,1094,13358,1249,7923,4547,17473,4700
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1523,2013,Olivos,Seguridad,540,486200,1,36530,77296,41612,35684,...,7599,22946,52574,1776,40877,2988,22731,11587,50064,15645
1524,2013,Vicente López,Movilidad,540,14164,1,12830,24790,13371,11419,...,1575,7292,16669,829,13177,844,7418,3631,15829,5330
1525,2013,Vicente López,Seguridad,540,50000,1,12830,24790,13371,11419,...,1575,7292,16669,829,13177,844,7418,3631,15829,5330
1526,2013,Vicente López,Deporte,540,147000,1,12830,24790,13371,11419,...,1575,7292,16669,829,13177,844,7418,3631,15829,5330


In [ ]:
df_definitivo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1528 entries, 0 to 1527
Data columns (total 30 columns):
 #   Column                                                                 Non-Null Count  Dtype 
---  ------                                                                 --------------  ----- 
 0   Año                                                                    1528 non-null   int64 
 1   Localidades                                                            1528 non-null   object
 2   Categoria                                                              1528 non-null   object
 3   Votos                                                                  1528 non-null   int64 
 4   Presupuesto                                                            1528 non-null   int64 
 5   Ganador_bin                                                            1528 non-null   int64 
 6   Viviendas particulares                                                 1528 non-null   int64 
 7

In [ ]:
X = df_definitivo.drop('Ganador_bin', axis=1)
y = df_definitivo['Ganador_bin']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
lista_numeric_features = df_definitivo.select_dtypes(include=['int64']).drop(['Ganador_bin'], axis=1).columns
lista_categorical_features = df_definitivo.select_dtypes(include=['object']).columns

In [ ]:
lista_numeric_features

Index(['Año', 'Votos', 'Presupuesto', 'Viviendas particulares',
       'Población en viviendas particulares', 'Mujer/Femenino',
       'Varón/Masculino', 'Red pública\n(agua corriente)',
       'Perforación con bomba a motor', 'Perforación con bomba manual',
       'Pozo sin bomba',
       'Transporte por cisterna, agua de lluvia, río, canal, arroyo o acequia',
       'Otra\nprocedencia', 'Tiene internet en la vivienda',
       'No tiene internet en la vivienda',
       'Obra social o\nprepaga\n(incluye PAMI)',
       'Programas o\nplanes estatales\nde salud',
       'No tiene obra\nsocial, prepaga ni\nplan estatal',
       'Población\ncon asistencia escolar',
       'Población\nsin asistencia escolar\npero que si asistió',
       'Población\nque nunca\ntuvo asistencia escolar',
       'Población económicamente activa ocupada',
       'Población económicamente activa desocupada',
       'Población no económicamente activa', '0-14', '15-64', '65-100 y más'],
      dtype='object')

In [ ]:
lista_categorical_features

Index(['Localidades', 'Categoria'], dtype='object')

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer()),
    ('scaler', StandardScaler())])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder())])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, lista_numeric_features),
        ('cat', categorical_transformer, lista_categorical_features)])

In [ ]:
preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer', SimpleImputer()),
                                                 ('scaler', StandardScaler())]),
                                 Index(['Año', 'Votos', 'Presupuesto', 'Viviendas particulares',
       'Población en viviendas particulares', 'Mujer/Femenino',
       'Varón/Masculino', 'Red pública\n(agua corriente)',
       'Perforación con bomba a motor', 'Perforación con bomba manual',
       'Pozo sin bomba',
       'Transp...
       'Población\nque nunca\ntuvo asistencia escolar',
       'Población económicamente activa ocupada',
       'Población económicamente activa desocupada',
       'Población no económicamente activa', '0-14', '15-64', '65-100 y más'],
      dtype='object')),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot', OneHotEncoder())]),
                                 Index(['Localidades', 'Categoria'], dtype='object'))])

In [ ]:
pipe = Pipeline(steps=[('preprocessor', preprocessor),
                      ('estimator', LogisticRegression())])

In [ ]:
def check_params_step(pipeline, params_keyword):
    all_params = pipeline.get_params().keys()
    available_params = [x for x in all_params if params_keyword in x]
    if len(available_params)==0:
        return "No matching params found!"
    else:
        return available_params

In [ ]:
check_params_step(pipe, 'imputer')


['preprocessor__num__imputer',
 'preprocessor__num__imputer__add_indicator',
 'preprocessor__num__imputer__copy',
 'preprocessor__num__imputer__fill_value',
 'preprocessor__num__imputer__keep_empty_features',
 'preprocessor__num__imputer__missing_values',
 'preprocessor__num__imputer__strategy',
 'preprocessor__cat__imputer',
 'preprocessor__cat__imputer__add_indicator',
 'preprocessor__cat__imputer__copy',
 'preprocessor__cat__imputer__fill_value',
 'preprocessor__cat__imputer__keep_empty_features',
 'preprocessor__cat__imputer__missing_values',
 'preprocessor__cat__imputer__strategy']

In [ ]:
params_grid = [

               {
                'preprocessor__num__imputer__strategy' : ['mean', 'median'],
                'estimator':[KNeighborsClassifier()],
                'estimator__n_neighbors': [ 10, 15, 20 ],

                },

               {
                'estimator': [DecisionTreeClassifier()],
                'estimator__max_leaf_nodes': [ 25, 50, 75],
                'preprocessor__num__imputer__strategy' : ['mean', 'median']
                }

               , {
                'estimator': [GaussianNB()],
                'preprocessor__num__imputer__strategy' : ['mean', 'median']
                }
              , {
                'estimator': [LogisticRegression()],
                'preprocessor__num__imputer__strategy' : ['mean', 'median']
                }
              ,
               {
                'estimator': [RandomForestClassifier()],
                'preprocessor__num__imputer__strategy' : ['mean', 'median'],
                'estimator__n_estimators': [ 100, 300, 500],
                'estimator__max_depth': [10, 15, 20]

                }

              ]


In [ ]:
skf = StratifiedKFold(n_splits=3)


In [ ]:
GS = GridSearchCV(pipe, params_grid, cv=skf)

GS.fit(X_train, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=3, random_state=None, shuffle=False),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer()),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         Index(['Año', 'Votos', 'Presupuesto', 'Viviendas particulares',
       'Población en viviendas particulares', 'Mujer/Femenino',
       'Varón/M...
                         {'estimator': [GaussianNB()],
                          'preprocessor__num__imputer__strategy': ['mean',
                                                                   'median']},
                         {'estimator': [LogisticRegression()],
                          'preprocessor__num__imputer__strategy': ['mean',
                                                                   'median']},
                         {'estimator': [RandomForestClassifier()],
                          'estimator__max_depth': [10, 15, 20],
                          'estimator__n_estimators': [100, 300, 500],
                          'preprocessor__num__imputer__strategy': ['mean',
                                                                   'median']}])

In [ ]:
pd.DataFrame(GS.cv_results_)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_estimator,param_estimator__n_neighbors,param_preprocessor__num__imputer__strategy,param_estimator__max_leaf_nodes,param_estimator__max_depth,param_estimator__n_estimators,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score
0,0.021371,0.008360,0.030715,0.025101,KNeighborsClassifier(),10.0,mean,NaN,NaN,NaN,"{'estimator': KNeighborsClassifier(), 'estimat...",0.764706,0.764128,0.751843,0.760225,0.005932,29
1,0.016354,0.000334,0.013621,0.001083,KNeighborsClassifier(),10.0,median,NaN,NaN,NaN,"{'estimator': KNeighborsClassifier(), 'estimat...",0.764706,0.764128,0.751843,0.760225,0.005932,29
2,0.016990,0.001817,0.015499,0.001490,KNeighborsClassifier(),15.0,mean,NaN,NaN,NaN,"{'estimator': KNeighborsClassifier(), 'estimat...",0.769608,0.769042,0.759214,0.765954,0.004772,25
3,0.015558,0.000286,0.013692,0.000520,KNeighborsClassifier(),15.0,median,NaN,NaN,NaN,"{'estimator': KNeighborsClassifier(), 'estimat...",0.769608,0.769042,0.759214,0.765954,0.004772,25
4,0.013715,0.000327,0.013656,0.000131,KNeighborsClassifier(),20.0,mean,NaN,NaN,NaN,"{'estimator': KNeighborsClassifier(), 'estimat...",0.747549,0.746929,0.737101,0.743860,0.004786,31
5,0.014801,0.000198,0.013251,0.000090,KNeighborsClassifier(),20.0,median,NaN,NaN,NaN,"{'estimator': KNeighborsClassifier(), 'estimat...",0.747549,0.746929,0.737101,0.743860,0.004786,31
6,0.017927,0.000373,0.008750,0.000608,DecisionTreeClassifier(),NaN,mean,25.0,NaN,NaN,"{'estimator': DecisionTreeClassifier(), 'estim...",0.816176,0.815725,0.783784,0.805228,0.015165,22
7,0.020590,0.000532,0.008406,0.000005,DecisionTreeClassifier(),NaN,median,25.0,NaN,NaN,"{'estimator': DecisionTreeClassifier(), 'estim...",0.816176,0.820639,0.781327,0.806047,0.017575,21
8,0.019154,0.000256,0.008810,0.000402,DecisionTreeClassifier(),NaN,mean,50.0,NaN,NaN,"{'estimator': DecisionTreeClassifier(), 'estim...",0.816176,0.823096,0.800983,0.813418,0.009236,15
9,0.022691,0.003327,0.010535,0.002399,DecisionTreeClassifier(),NaN,median,50.0,NaN,NaN,"{'estimator': DecisionTreeClassifier(), 'estim...",0.808824,0.823096,0.800983,0.810967,0.009154,16


In [ ]:
grid=pd.DataFrame(GS.cv_results_)

In [ ]:
grid.sort_values('rank_test_score')

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_estimator,param_estimator__n_neighbors,param_preprocessor__num__imputer__strategy,param_estimator__max_leaf_nodes,param_estimator__max_depth,param_estimator__n_estimators,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score
27,1.313509,0.104128,0.073622,0.000753,RandomForestClassifier(),NaN,median,NaN,15.0,500.0,"{'estimator': RandomForestClassifier(), 'estim...",0.811275,0.823096,0.830467,0.821612,0.007905,1
19,0.699481,0.014694,0.047752,0.003601,RandomForestClassifier(),NaN,median,NaN,10.0,300.0,"{'estimator': RandomForestClassifier(), 'estim...",0.818627,0.813268,0.832924,0.821606,0.008296,2
22,0.265573,0.008048,0.023086,0.001684,RandomForestClassifier(),NaN,mean,NaN,15.0,100.0,"{'estimator': RandomForestClassifier(), 'estim...",0.801471,0.820639,0.840295,0.820801,0.015850,3
32,3.323803,0.561845,0.207606,0.049455,RandomForestClassifier(),NaN,mean,NaN,20.0,500.0,"{'estimator': RandomForestClassifier(), 'estim...",0.811275,0.820639,0.830467,0.820793,0.007836,4
31,0.860874,0.115213,0.053563,0.006572,RandomForestClassifier(),NaN,median,NaN,20.0,300.0,"{'estimator': RandomForestClassifier(), 'estim...",0.803922,0.823096,0.832924,0.819980,0.012043,5
30,0.802897,0.065677,0.058741,0.017949,RandomForestClassifier(),NaN,mean,NaN,20.0,300.0,"{'estimator': RandomForestClassifier(), 'estim...",0.806373,0.818182,0.830467,0.818340,0.009837,6
24,0.744482,0.017685,0.050903,0.005981,RandomForestClassifier(),NaN,mean,NaN,15.0,300.0,"{'estimator': RandomForestClassifier(), 'estim...",0.811275,0.823096,0.820639,0.818336,0.005093,7
25,0.734047,0.008191,0.053588,0.005529,RandomForestClassifier(),NaN,median,NaN,15.0,300.0,"{'estimator': RandomForestClassifier(), 'estim...",0.803922,0.815725,0.832924,0.817523,0.011908,8
33,2.327665,1.201261,0.084535,0.012006,RandomForestClassifier(),NaN,median,NaN,20.0,500.0,"{'estimator': RandomForestClassifier(), 'estim...",0.808824,0.815725,0.828010,0.817519,0.007935,9
20,1.173618,0.014826,0.070092,0.006169,RandomForestClassifier(),NaN,mean,NaN,10.0,500.0,"{'estimator': RandomForestClassifier(), 'estim...",0.818627,0.815725,0.815725,0.816692,0.001368,10


In [ ]:
GS.best_estimator_

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['Año', 'Votos', 'Presupuesto', 'Viviendas particulares',
       'Población en viviendas particulares', 'Mujer/Femenino',
       'Varón/Masculino', 'Red pública\n(agua corriente)',
       'Perforación con bomba a motor', '...
       'Población económicamente activa ocupada',
       'Población económicamente activa desocupada',
       'Población no económicamente activa', '0-14', '15-64', '65-100 y más'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder())]),
                                                  Index(['Localidades', 'Categoria'], dtype='object'))])),
                ('estimator',
                 RandomForestClassifier(max_depth=15, n_estimators=500))])

In [ ]:
GS.best_params_

{'estimator': RandomForestClassifier(),
 'estimator__max_depth': 15,
 'estimator__n_estimators': 500,
 'preprocessor__num__imputer__strategy': 'median'}

In [ ]:
modelo_final=GS.best_estimator_

In [ ]:
modelo_final.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['Año', 'Votos', 'Presupuesto', 'Viviendas particulares',
       'Población en viviendas particulares', 'Mujer/Femenino',
       'Varón/Masculino', 'Red pública\n(agua corriente)',
       'Perforación con bomba a motor', '...
       'Población económicamente activa ocupada',
       'Población económicamente activa desocupada',
       'Población no económicamente activa', '0-14', '15-64', '65-100 y más'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder())]),
                                                  Index(['Localidades', 'Categoria'], dtype='object'))])),
                ('estimator',
                 RandomForestClassifier(max_depth=15, n_estimators=500))])

In [ ]:
modelo_final.score(X_test,y_test)

0.8169934640522876

In [ ]:
# Probar bloque nuevo. PROBAR TODO EL CODIGO HASTA ACA

In [ ]:
# ==============================================================================
# FUNCIÓN PARA PREDECIR SI UN NUEVO PROYECTO SERÁ APROBADO O NO
# ==============================================================================

#def predecir_aprobacion_proyecto(año, localidad, categoria, presupuesto, modelo, df_censo):
    """
    Toma los datos de un nuevo proyecto, le cruza la información del censo correspondiente
    a su localidad y utiliza el modelo preentrenado para predecir el resultado.
    """
    # 1. Crear un diccionario con los datos básicos del nuevo proyecto
    #nuevo_proy_dict = {
        'Año': [año],
        'Localidades': [localidad],
        'Categoria': [categoria],
        'Presupuesto': [presupuesto]
    }
    #nuevo_df = pd.DataFrame(nuevo_proy_dict)

    # 2. Hacer el join con el censo para traer todas las variables que el modelo requiere
    # (Asegúrate de que df_censo sea el dataframe original del Censo 2022 por localidad)
    nuevo_df_completo = pd.merge(nuevo_df, df_censo, on='Localidades', how='left')

    # 3. Validar que la localidad exista en los datos del censo
    #if nuevo_df_completo.isnull().values.any():
        #print(f"⚠️ Advertencia: Algunos datos demográficos no se encontraron para la localidad '{localidad}'. Verifique que esté bien escrita.")

    # 4. Realizar la predicción utilizando el Pipeline entrenado
    # El pipeline se encarga automáticamente de hacer el Imputer, OneHotEncoder y StandardScaler
    #prediccion = modelo.predict(nuevo_df_completo)
    #probabilidad = modelo.predict_proba(nuevo_df_completo)[0][1] # Probabilidad de que sea exitoso (clase 1)

    # 5. Formatear y retornar el resultado
    #if prediccion[0] == 1:
        #resultado = "APROBADO (Ganador)"
    #else:
        #resultado = "NO APROBADO"

    #print(f"🔮 Predicción para el proyecto: {resultado} (Probabilidad de éxito: {probabilidad:.2%})")
    #return prediccion[0]

# ==============================================================================
# EJEMPLO DE USO: Carga aquí los datos del nuevo proyecto que quieras evaluar
# ==============================================================================

# Supongamos que tu mejor modelo guardado de la optimización por GridSearch se llama 'modelo_final'
# y tu dataframe del censo limpio se llama 'df_censo_2022'

#nuevo_proyecto_ejemplo = {
    'año': 2026,
    'localidad': 'Olivos',               # Asegúrate de usar nombres exactos como figuran en tu dataset
    'categoria': 'Seguridad',            # Asegúrate de usar categorías exactas (ej: 'Seguridad', 'Educación')
    'presupuesto': 5000000,              # Monto estimado en pesos
    'modelo': modelo_final,              # Tu Pipeline ajustado (ej: RandomForest o el mejor del GridSearch)
    'df_censo': df_censo_2022            # El DataFrame del censo con la clave 'Localidades'
}

# Ejecutar la predicción
#prediccion_final = predecir_aprobacion_proyecto(**nuevo_proyecto_ejemplo)

In [ ]:
#import joblib
#joblib.dump(modelo_final, 'modelo_presupuesto_participativo.pkl')